# Install dependencies (only needed if running on Colab). torch is not needed here.
!pip install -q "typesafe-sdk>=0.7.5" openpyxl pandas scikit-learn numpy


In [ ]:
# Get the project. Idempotent: safe to re-run after a runtime restart.
# The datasets are tracked in the repo, so the clone brings code + data.
import os, subprocess, pathlib

if not pathlib.Path("NLP").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iammatthi/NLP.git"], check=True)
os.chdir("NLP")
print("working dir:", os.getcwd())

for rel in (
    "data/raw/depression-severity-bengali.xlsx",
    "data/raw/depression-severity-english.csv",
):
    data_file = pathlib.Path(rel)
    assert data_file.exists(), f"{data_file} missing - the clone did not bring the data"
    print("dataset OK:", data_file, f"({data_file.stat().st_size:,} bytes)")


In [ ]:
# Install dependencies (only needed if running on Colab). torch is not needed here.
!pip install -q "typesafe-sdk>=0.7.5" openpyxl pandas scikit-learn numpy


In [ ]:
# Configure TYPESAFE_API_KEY from local .env, environment, or Colab Secrets.
import os
from src.jev_client import load_env_key

api_key = load_env_key("TYPESAFE_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("TYPESAFE_API_KEY")
        if api_key:
            os.environ["TYPESAFE_API_KEY"] = api_key
            print("TYPESAFE_API_KEY loaded from Colab Secrets")
    except Exception:
        pass

assert os.environ.get("TYPESAFE_API_KEY"), (
    "TYPESAFE_API_KEY not found. Set it in Repository/.env or Colab Secrets."
)
print("TYPESAFE_API_KEY configured: OK")

In [ ]:
# Prepare stratified 80/10/10 splits (seed=42) and run offline + live 2-row smoke check.
!python scripts/prepare_splits.py --lang bn --seed 42
!python scripts/prepare_splits.py --lang en --seed 42
!python scripts/smoke_test_jev.py --live

In [ ]:
# Configuration -- single source of truth for the Jev evaluation sweep.
SEED = 42
CONCURRENCY = 10
BOOTSTRAP = 1000
RUNS = [("bn", "unified"), ("en", "unified"), ("en", "naseem_dda")]  # (lang, rubric)
print(f"seed={SEED} concurrency={CONCURRENCY} bootstrap={BOOTSTRAP} runs={RUNS}")


## Run Zero-Shot Evaluation on `bn-test` ($n=490$) and `en-test` ($n=356$)

`evaluate_jev.py` queries both `Choice` and `Score` in a single request per item, caches raw responses in `outputs/experiment6/`, and writes `eval_test.json` reports to `results/experiment6/`.

In [ ]:
# Zero-shot evaluation sweep (runs from the Configuration cell).
import subprocess

for lang, rubric in RUNS:
    subprocess.run(
        ["python", "evaluate_jev.py", "--lang", lang, "--split", "test",
         "--rubric", rubric, "--concurrency", str(CONCURRENCY),
         "--bootstrap", str(BOOTSTRAP), "--seed", str(SEED)],
        check=True,
    )


## Summary Table

In [ ]:
import json
from pathlib import Path

configs = [
    ("6a Choice (BN unified)", "results/experiment6/6a_jev_choice_bn/eval_test.json"),
    ("6b Score  (BN unified)", "results/experiment6/6b_jev_score_bn/eval_test.json"),
    ("6a Choice (EN unified)", "results/experiment6/6a_jev_choice_en/eval_test.json"),
    ("6b Score  (EN unified)", "results/experiment6/6b_jev_score_en/eval_test.json"),
    ("6c Choice (EN naseem_dda)", "results/experiment6/6c_jev_choice_naseem_dda_en/eval_test.json"),
    ("6c Score  (EN naseem_dda)", "results/experiment6/6c_jev_score_naseem_dda_en/eval_test.json"),
]

print(f"{'Configuration':26s} | {'macro-F1':8s} | {'QWK':7s} | {'MAE':7s} | {'Acc':7s} | {'Down%':7s} | {'Up%':7s} | {'FScore':7s} | {'min/mild/mod/sev F1'}")
print("-" * 120)
for label, rel_path in configs:
    p = Path(rel_path)
    if not p.exists():
        continue
    r = json.loads(p.read_text(encoding="utf-8"))
    d = r["direction"]
    g = r["graded"]
    cls_f1 = " / ".join(f"{r[c]:.4f}" for c in ("minimal", "mild", "moderate", "severe"))
    print(
        f"{label:26s} | {r['macro_f1']:8.4f} | {r['qwk']:7.4f} | {r['mae']:7.4f} | "
        f"{r['accuracy']:7.4f} | {d['downgrade_rate']:7.4f} | {d['upgrade_rate']:7.4f} | "
        f"{g['graded_fscore']:7.4f} | {cls_f1}"
    )